In [1]:
# https://ai.google.dev/gemini-api/docs/live-api/get-started-sdk#python_1
# https://googleapis.github.io/python-genai/genai.html#module-genai.live

In [2]:
import asyncio
import os

import matplotlib.pyplot as plt
import numpy as np
import sounddevice as sd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Audio

load_dotenv()

True

# InputStream

In [ ]:
SAMPLE_RATE = 16_000

chunks = []

def callback(indata, frames, time_info, status):
    chunks.append(indata.reshape(-1).copy())

with sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype=np.int16,
    callback=callback,
):
    sd.sleep(3000)
    
chunks_np = np.concat(chunks).astype("<i2")

Producer gonna put a chunk
1/10, got chunk of length 1024
Producer gonna put a chunk
2/10, got chunk of length 1024
Producer gonna put a chunk
3/10, got chunk of length 1024
Producer gonna put a chunk
4/10, got chunk of length 1024
Producer gonna put a chunk
5/10, got chunk of length 1024
Producer gonna put a chunk
6/10, got chunk of length 1024
Producer gonna put a chunk
7/10, got chunk of length 1024
Producer gonna put a chunk
8/10, got chunk of length 1024
Producer gonna put a chunk
9/10, got chunk of length 1024
Producer gonna put a chunk
10/10, got chunk of length 1024


In [48]:
chunks_bytes = chunks_np.tobytes()
Audio(chunks_np, rate=SAMPLE_RATE)

# Send over to Gemini

In [ ]:
received_bytes = bytearray()

API_KEY = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

model = "gemini-3.8-live"
config = {"response_modalities": ["AUDIO"]}


async def main():
    async with client.aio.live.connect(model=model, config=config) as session:
        await session.send_realtime_input(
            audio=types.Blob(
                data=chunks_bytes,
                mime_type="audio/pcm;rate=16000",
            ),
        )

        async for response in session.receive():
            content = response.server_content
            if content is None:
                continue

            if content.model_turn:
                for part in content.model_turn.parts:
                    if part.inline_data:
                        received_bytes.extend(part.inline_data.data)

            if content.input_transcription:
                print(f"User: {content.input_transcription.text}")

            if content.output_transcription:
                print(f"Gemini: {content.output_transcription.text}")


await main()

User: Hello.
Gemini: Hello! How can I 
Gemini: help you today?


In [50]:
Audio(np.frombuffer(received_bytes, dtype="<i2"), rate=24000)